In [ ]:
# from ultralytics import YOLO
# from glob import glob
# from PIL import Image
# from PIL import ImageDraw

In [ ]:
# model = YOLO('yolov8x-seg.pt')

In [ ]:
# res = model(glob('calib/*.JPG'))

In [ ]:
# ims = []
# masks = res[3].masks
# mask1 = masks[1]
# mask = mask1.data[0].cpu().numpy()
# polygon = mask1.xy[0]
# mask_img = Image.fromarray(mask, "I")
# mask_img


In [ ]:
# img = Image.open(res[3].path)
# draw = ImageDraw.Draw(img)
# draw.polygon(polygon,outline=(0,255,0), width=5)
# img

In [ ]:
%cd calib

In [ ]:
import ultralytics
from IPython.display import display, Image
from roboflow import Roboflow
import cv2
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set the device for GPU acceleration
device = "cuda"

# Check Ultralytics version and setup completion
ultralytics.checks()

# Set the first_run flag to False after the initial run
first_run = False

In [ ]:
# if first_run:
#     !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/segment-anything.git'
#     !wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

In [ ]:
from ultralytics import YOLO

# Load the YOLOv8 model
model = YOLO('yolov8x.pt')

image = '/home/biodiv/calib/human_images/cam2 02-10-2024/03.JPG'
# Perform object detection on the image
results = model.predict(source=image, conf=0.25, classes=[0])

In [ ]:
# results = model.predict(source=image, conf=0.25, classes=[0])
for result in results:
    boxes = result.boxes

bbox = result.boxes.xyxy.tolist()[0]

In [ ]:
bbox

In [ ]:
def show_mask(mask, ax, random_color=False):

    color = np.array([255, 255, 255])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    
def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   
    
def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0,0,0,0), lw=2))    

In [ ]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)


In [ ]:
ims = glob(f'{Path(image).parent}/*')
ims.sort()
ims

In [ ]:
for im_path in ims:

    results = model(im_path, classes=[0])
    for result in results:
        boxes = result.boxes

    if not result.probs:
        continue

    # if im_path in ['d/images/24.JPG', 'd/images/21.JPG', 'd/images/18.JPG']:
    #     bbox = boxes.xyxy.tolist()[1]
    # else:
    
    bbox = boxes.xyxy.tolist()
    if not bbox:
        print('Could not find a person')
        continue
    else:
        bbox = bbox[0]

    image = cv2.cvtColor(cv2.imread(im_path), cv2.COLOR_BGR2RGB)
    predictor.set_image(image)
    
    input_box = np.array(bbox)
    masks, _, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box,
        multimask_output=False,
    )
    
    segmentation_mask = masks[0]
    
    # Convert the segmentation mask to a binary mask
    binary_mask = np.where(segmentation_mask > 0.5, 1, 0)
    
    # Create white background and black background
    white_background = np.ones_like(image) * 255
    black_background = np.zeros_like(image)
    
    # Apply the binary mask to the white background
    new_image = white_background * binary_mask[..., np.newaxis] + black_background * (1 - binary_mask[..., np.newaxis])
    
    # Display the image
    plt.imshow(new_image.astype(np.uint8))
    plt.axis('off')
    plt.show()
    
    # Save the image
    plt.imsave('/home/biodiv/calib/human_images/cam2 02-10-2024/masks/' + Path(im_path).name, new_image.astype(np.uint8))

In [ ]:
# segmentation_mask = masks[0]

# # Convert the segmentation mask to a binary mask
# binary_mask = np.where(segmentation_mask > 0.5, 1, 0)

# # Create white background and black background
# white_background = np.ones_like(image) * 255
# black_background = np.zeros_like(image)

# # Apply the binary mask to the white background
# new_image = white_background * binary_mask[..., np.newaxis] + black_background * (1 - binary_mask[..., np.newaxis])

# # Display the image
# plt.imshow(new_image.astype(np.uint8))
# plt.axis('off')
# plt.show()

# # Save the image
# plt.imsave("masked_image.png", new_image.astype(np.uint8))
# cv2.imwrite('output_image.jpg', image)